# Hands-On AI for Science
## August 14, Afternoon: When This Works and When It Doesn't

### Self-contained Colab notebook

Everything this notebook needs is either in it or downloaded by its own cells. Just run the cells in order. No GPU is required.


Thursday morning, we asked: *does the model work on data it hasn't seen?*  The way we evaluate this is to use accuracy on held-out items. This final session asks a harder question: **when is held-out accuracy itself misleading?**

Every kind of failure in this hour has appeared in published, peer-reviewed work. These are not hypothetical mistakes. A result that survives this session's checklist is not foolproof. But it is one that can be defended.

## Setup

Everything in this session runs offline on files already in the repository — no model downloads, no network.  If a file prints MISSING, Jupyter was probably started outside the `lectures` folder.

### Data files

Run this cell to download the data files this notebook uses. You do not need to edit it.

In [ ]:
# The data files this notebook uses, fetched from the workshop repository.
# Run this cell once; you do not need to edit it.
import os, urllib.request

_FILES = [
    "protein_localization.csv",
    "protein_embeddings.npy",
    "blood_cells.npz",
    "image_embeddings.npy",
    "CHECKLIST.md",
]

for _name in _FILES:
    if not os.path.exists(_name):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/jhasegaw/hands_on_ai_for_science/main/lectures/' + _name, _name)
print('Ready:', ', '.join(_FILES))

In [ ]:
import importlib.util, os

for pkg in ['numpy', 'pandas', 'matplotlib', 'sklearn', 'scipy']:
    print(f'{pkg:26s}', 'OK' if importlib.util.find_spec(pkg) is not None else 'MISSING')
for f in ['protein_localization.csv', 'protein_embeddings.npy', 'blood_cells.npz',
          'image_embeddings.npy', 'CHECKLIST.md']:
    print(f'{f:26s}', 'OK' if os.path.exists(f) else 'MISSING')


1. [The ways held-out accuracy deceives](#taxonomy)
1. [Leakage, measured on our own data](#leakage)
1. [What these models are, and what they discover](#bigpicture)
1. [The checklist](#checklist)
1. [Homework: three diagnostics for your own data](#homework)
1. [Where to go next](#next)
1. [Appendix (optional): batch effects, live](#batch)

<a id="taxonomy"></a>

## 1. The ways held-out accuracy deceives

Five failure modes cover most of the mistakes that get made with AI and machine learning.  Each has a documented case in the scientific literature *(citations at the end of the notebook)*.

**(1) Leakage.**  Information from the test data reaches the model — through near-duplicate samples straddling the split (homologous sequences, repeated subjects, resampled images), or through preprocessing fit on all the data.  Thursday's sessions demonstrated both mechanisms; Section 2 measures them on this workshop's own datasets.

**(2) Confounds and batch effects becoming the signal.**  The model learns the *measurement*, not the biology.  The canonical case: a pneumonia detector that worked partly by recognizing **which hospital** an X-ray came from. For example, portable X-ray machines, used at sicker patients' bedsides, left visible signatures in the images (Zech et al. 2018).  The model was never wrong about the correlation. The correlation was just not pneumonia.

**(3) Shortcut learning.**  A general name for the pattern that models take the path of least resistance to the training objective (Geirhos et al. 2020).  Melanoma classifiers reached dermatologist-level benchmarks partly by noticing **surgical skin markings** near lesions that surgeons had already judged suspicious (Winkler et al. 2019).  The shortcut *was* predictive. That is what makes shortcuts dangerous.

**(4) Distribution shift.**  Everything this week assumed we are testing data that is *like the training data*.  A new lab, instrument, cohort, or season is a different distribution, and performance there is a new empirical question, not an extrapolation you get for free.

**(5) Underpowered claims.**  Small test sets make noisy numbers (30 samples ≈ ±15 points), and "we need more data" is itself a testable claim.  The tool is the **learning curve** — accuracy as a function of training-set size, one plot:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, KFold, cross_val_score

proteins = pd.read_csv('protein_localization.csv')
embeddings = np.load('protein_embeddings.npy')
y = proteins['location'].to_numpy()
families = proteins['uniref50'].to_numpy()
clf = LogisticRegression(max_iter=2000)

sizes = [50, 100, 200, 300, 450]
perm = np.random.default_rng(0).permutation(len(y))
accs = []
for n in sizes:
    idx = perm[:n]
    accs.append(cross_val_score(clf, embeddings[idx], y[idx],
                                cv=GroupKFold(5), groups=families[idx]).mean())

fig = plt.figure(figsize=(8, 4))
ax = fig.subplots(1)
ax.plot(sizes, accs, 'o-')
ax.set_xlabel('training samples'); ax.set_ylabel('grouped CV accuracy')
ax.set_ylim(0.5, 1.0); ax.set_title('Learning curve: the morning protein probe')
for n, a in zip(sizes, accs): print(f'n={n:4d}: {a:.3f}')

In this case, how much does sample size matter? The curve is **nearly flat** — about 0.88 with only 50 proteins, 0.90 with 450.  There are two lessons here. First, in this case, collecting another 500 proteins of the same kind would mostly buy nothing (the remaining errors need something else — a bigger model, better labels, different features). Second, our earlier claim that frozen foundation-model features "work with ~50 labeled examples" is correct in this case. A different shape — still climbing at the largest n — would be evidence that more data would be useful.

<a id="leakage"></a>

## 2. Leakage, measured on our own data

This morning's probe used protein embeddings to predict where each protein lives in the cell. With an honest split, it scored about 90% accuracy. We will now run three checks on that result. Together, they answer a simple question: *how much of that 0.90 was real?*

### Check 1: Compare a random split with a grouped split

First, a reminder of the two ways data can be split.

* A **random split** assigns every sample to the training set or the test set at random. This is the default in most software.
* A **grouped split** first gathers related samples into groups. Here, a group is a protein family (the `uniref50` column in our dataset). The split then keeps each whole group on one side. Related proteins are never divided between training and test.

Thursday's rule was: when related samples exist, evaluate both ways and compare the two numbers. If the random split scores much higher than the grouped split, the extra accuracy did not come from biology. It came from the model recognizing near-copies of its training samples in the test set.

Here is that comparison, for this morning's probe:

In [ ]:
acc_random  = cross_val_score(clf, embeddings, y, cv=KFold(5, shuffle=True, random_state=0)).mean()
acc_grouped = cross_val_score(clf, embeddings, y, cv=GroupKFold(5), groups=families).mean()
print(f'random split:  {acc_random:.3f}')
print(f'family split:  {acc_grouped:.3f}')

There is no gap between the two numbers.  This dataset was deliberately curated — mostly one protein per family — so there was little leakage *to* find.  **We can be confident a dataset is clean when the family split and the random split agree.**  A large gap here would have meant the random-split number was inflated by the model recognizing near-copies. No gap means the 0.90 is more likely to be real. The test was two lines of code, and it is always a good idea to run it.

### Check 2: Look at what a random split actually tests

Our dataset does contain one set of near-duplicates: the interferon-alpha family. These are immune-signaling proteins whose sequences are nearly identical to one another. They give us a close-up view of what each kind of split really measures.

Take one family member, IFNA1, and put it in the test set. Now ask: what is the closest protein remaining in the training set? We measure closeness as cosine distance between embeddings, where 0 means identical. The next cell answers the question twice. Once for a random split, where other family members usually stay in the training set. And once for a family split, where the whole family leaves the training set together:

In [ ]:
from scipy.spatial.distance import pdist, squareform

D = squareform(pdist(embeddings, 'cosine'))
genes = list(proteins['gene'])
i = genes.index('IFNA1')

others = [j for j in range(len(genes)) if j != i]
nn = min(others, key=lambda j: D[i, j])
no_family = [j for j in range(len(genes)) if not genes[j].startswith('IFNA')]
nn_honest = min(no_family, key=lambda j: D[i, j])

print(f'random split -- nearest training neighbor:      {genes[nn]:8s} distance {D[i, nn]:.4f}')
print(f'family held out -- nearest training neighbor:   {genes[nn_honest]:8s} distance {D[i, nn_honest]:.4f}')

The two numbers tell the story. Under the random split, the nearest training example sits at distance 0.0002. That is effectively the same protein. To classify it correctly, the model does not need to have learned anything about localization. It only needs to give the same answer to the same input. Under the family split, the nearest training example is 80 times farther away. Now the test protein is genuinely new, and a correct answer is evidence of real generalization.

One caution comes with this example. The nearest protein outside the family, IFNW1, is still a relative (interferon-omega). Relatedness is a matter of degree, so every grouping variable is an approximation. The homework function `check_duplicates` exists for exactly this reason.

### Check 3: Measure how redundancy inflates accuracy

Why did Check 1 find no gap in our dataset? Because the gap depends on how much **redundancy** a dataset contains. Redundancy means near-duplicate samples. Copies, close relatives, repeated measurements of the same thing.

Our dataset has little redundancy, because it was built that way. Most real data collections have a lot. The simulation below shows what that does to the two splits. We start from our clean dataset. We then imitate a real collection effort by adding near-copies of some proteins. Each copy is the original embedding plus a small amount of noise. The noise is set so that a copy is about as close to its original as real family members are to each other. At each level of copying, we evaluate with both splits.

This is a constructed simulation, like Thursday's simulated mice, and is labeled as such:

In [ ]:
rng = np.random.default_rng(0)
scale = np.linalg.norm(embeddings, axis=1).mean()
dup_targets = rng.choice(len(y), 90, replace=False)

dup_levels = [0, 1, 2, 4, 8]
rand_accs, grp_accs = [], []
for d in dup_levels:
    Xs, ys, gs = [embeddings], [y], [np.arange(len(y))]
    for _ in range(d):
        noise = rng.normal(0, 0.002 * scale, (len(dup_targets), embeddings.shape[1])).astype(np.float32)
        Xs.append(embeddings[dup_targets] + noise)
        ys.append(y[dup_targets]); gs.append(dup_targets)
    X2, y2, g2 = np.concatenate(Xs), np.concatenate(ys), np.concatenate(gs)
    rand_accs.append(cross_val_score(clf, X2, y2, cv=KFold(5, shuffle=True, random_state=0)).mean())
    grp_accs.append(cross_val_score(clf, X2, y2, cv=GroupKFold(5), groups=g2).mean())
    print(f'copies per duplicated protein: {d}  ->  random {rand_accs[-1]:.3f}   grouped {grp_accs[-1]:.3f}')

fig = plt.figure(figsize=(8, 4))
ax = fig.subplots(1)
ax.plot(dup_levels, rand_accs, 'o-', label='random split')
ax.plot(dup_levels, grp_accs, 's-', label='grouped split (honest)')
ax.set_xlabel('near-copies added per duplicated protein'); ax.set_ylabel('CV accuracy')
ax.legend(); ax.set_title('Redundancy inflates the random split, and only the random split');

To read this plot, keep in mind what the two lines are.

* The **grouped split** line is the honest measurement. Related samples, including our artificial near-copies, always stay on the same side of the split. This line stays flat near 0.91 no matter how many copies are added. That is correct behavior. Adding copies of proteins we already have adds no new information, so the true difficulty of the task has not changed.
* The **random split** line ignores relatedness. As copies accumulate, more and more test proteins have a near-copy sitting in the training set. The score climbs toward 0.97. The dataset is being rewarded for containing copies of itself.

Real data collections have exactly this kind of redundancy. Sequence databases contain families of related proteins. Image collections contain near-duplicate photos. Clinical archives contain repeat visits from the same patients. The practical rule: **when a model is evaluated with a random split on data drawn from a public database, treat the reported accuracy as an upper bound, not as a measurement.**

<a id="bigpicture"></a>

## 3. What these models are, and what they discover

Now, a little bit without much code, on the questions people actually ask about AI. We will use what this workshop built as the evidence.

### "LLMs just copy their training data" or are "Stochastic Parrots".

This claim is testably false, and the arithmetic alone shows it. Large models are trained on far more data than their parameters could store verbatim.  Training is **compression**, and compression forces extraction of regularities. Large language models can and do generate sentences they have never seen before. And our previous demonstrations showed this in protein models using similar architectures. ESM-2's embeddings organized proteins by function and localization it was never told about, and DINOv2's organized blood cells by morphology. Probes decode information that could not be "copied" from labels that were never present.

But this cuts both ways.  Models **do** memorize some training items verbatim, especially repeated ones. For science this has a concrete consequence we saw twice today. **If an evaluation item existed publicly (UniProt, GenBank, the open web), a pretrained model may have effectively seen it.**  Benchmark contamination is a variant of the memorization debate.  The truthful summary of what a model "knows" is a spectrum — memorization, interpolation, and abstraction. And *for any single prediction, where it sits on that spectrum is usually unknown.*  Much of this session is about behaving sensibly under that uncertainty.

### Generalization — and its edge

Within the training distribution, generalization is real; we demonstrated it repeatedly.  Outside the distribution, the default answer is *no*.  But there is a deep and open middle case: sometimes a model recovers something like the **underlying generating process**, and then limited extrapolation becomes possible.  The workshop's own toy problem makes this visible.  Thursday, a degree-3 polynomial was fit to 24 noisy points of a sine — and a degree-15 polynomial to the same points.  Both fit the training interval.  Watch what happens *outside* it:

In [ ]:
def fit_poly(x, yy, degree):
    H = np.stack([x**k for k in range(degree + 1)], axis=1)
    return np.linalg.lstsq(H, yy, rcond=None)[0]
def poly_predict(c, x):
    return sum(ci * x**k for k, ci in enumerate(c))

rng1 = np.random.default_rng(1)
x_train = np.sort(rng1.uniform(-np.pi, np.pi, 24))
y_train = np.sin(x_train) + rng1.normal(0, 0.25, 24)

xw = np.linspace(-1.6 * np.pi, 1.6 * np.pi, 600)
fig = plt.figure(figsize=(11, 4))
ax = fig.subplots(1)
ax.axvspan(-np.pi, np.pi, color='0.92', label='training region')
ax.plot(xw, np.sin(xw), 'k--', label='the truth')
ax.plot(x_train, y_train, 'o', markersize=6)
for deg, style in [(3, '-'), (15, '-')]:
    ax.plot(xw, poly_predict(fit_poly(x_train, y_train, deg), xw), style, label=f'degree {deg}')
ax.set_ylim(-3, 3); ax.legend(fontsize=11);

Inside the shaded region, the two models are hard to tell apart.  Outside it, the degree-15 model — which merely *fit* — explodes immediately, while the degree-3 model — which approximately **recovered the generating function** — degrades gracefully and even tracks the next arch of the sine for a while. But it too eventually goes awry.  Limited extrapolation is possible exactly to the extent that the model has captured the process rather than the samples.

Similar results have been found for how LLMs do arithmetic with large numbers. Suppose a model is trained on additions like 3+2=5, 15+2=17, and 458+2=460. One function fits all of that data perfectly: y = x + 2. But suppose the model instead learns y = 1.00000001·x + 2. On every example it was trained on, the two functions give the same answer after rounding. The tiny error in the slope is invisible. Training cannot tell the two functions apart, so the model has no reason to prefer the exact rule. Now ask for 1,000,000,000 + 2. The learned function returns 1,000,000,012. The error was always there. It only becomes visible outside the range the training data covered. (The function a real LLM learns is more complicated than a line, but the failure has the same shape: correct inside the training range, drifting outside it.) Real LLMs show exactly this signature. Addition is near-perfect on numbers like those seen in training, with accuracy decaying as the numbers grow — the same accuracy-by-digit-count decay measured in this morning's addition model.

Does this happen in large neural networks?  Sometimes, demonstrably: in the **grokking** phenomenon (Power et al. 2022), small transformers trained on modular arithmetic first memorize their training table — and then, abruptly, late in training, discover the *actual algorithm* and generalize perfectly.  The transition from memorization to mechanism is real.  What is missing is any general way to know, for a given model and question, whether it has happened.  "Did AlphaFold learn something true about protein physics, or a spectacularly good interpolation of known structures?" is a live research question. And it is *this* question, asked of a specific model.  Scientists who can probe, ablate, and design controlled comparisons are precisely the people equipped to answer it, one model and one claim at a time.

### This explains the AI news

Two famous "mysteries" stop being mysterious once a model is seen as a fitted statistical object rather than a reasoner:

* **Model hallucination is not a malfunction.**  A generative model produces *plausible* continuations — in-distribution outputs — not retrieved facts.  Where its training data is dense, plausible and true mostly coincide; where data is thin, the model produces the same fluent confidence with nothing underneath.  Fluency is not evidence.  The tiny addition model from this morning did exactly this, generating confidently wrong sums.
* **Adversarial fragility is shortcut learning, inverted.**  If a model leans on statistically-predictive-but-shallow features (textures, stains, pixel statistics), then tiny, targeted nudges to those features flip its answers while changing nothing a human or model with deeper understanding sees.  The melanoma-marking study and the famous imperceptibly-perturbed images are the same phenomenon at different scales: the model never saw what we assumed it saw.

### Why deployed assistants seem to escape this — and why your models won't

Systems like Claude and ChatGPT are not raw models; they are **engineered systems built around models**, with layers that exist precisely because of everything above. They use retrieval that grounds answers in actual documents. They have **tool calls that hand arithmetic, code, and lookups to symbolic engines that cannot hallucinate**. They use chain-of-thought inference that spends compute decomposing problems — the same reasoning-trace idea trained into this morning's addition model. And they have verification and feedback layers tuning the model toward calibration and refusal.  Hybrid neuro-symbolic designs (a language model proposing, a symbolic system checking — e.g. AlphaGeometry) push this further.

Two consequences matter for scientists:

1. **These layers mitigate; they do not solve.**  The checklist questions apply to assistant output too — provenance, contamination, confident-but-thin territory.
2. **Everything used in this workshop is the raw object.**  ESM-2, DINOv2, any model downloaded from a hub and run locally: no retrieval, no verifier, no guardrails.  The failure modes in today's taxonomy are unmanaged by default, and managing them is the user's job.

Which is exactly what the next section is: that job, written down.

### And when "why?" is the question

The instinct after any striking result is *can the model tell me why?*  Mostly, no — and the tools that seem to answer it (saliency maps, attention visualizations) are less reliable than advertised; treat them as hypothesis generators at best.  What does work is what this room already knows how to do: **designed experiments, run on the model.**  Probe for information (Thursday).  Ablate inputs and see what breaks.  Compare controlled variants.  Treat the model as an organism that can be experimented on — cheaply, repeatably, at scale.  That reframing is the workshop's closing argument: the skills of experimental science are not being replaced by these models; they are what these models have been missing.

<a id="checklist"></a>

## 4. The checklist

The file **`CHECKLIST.md`** in this folder is a take-home artifact: one page, six sections — the number itself, leakage, independence, confounds, scope, and interpretation — each item phrased as a question with the cheapest way to check it.  It condenses Thursday morning's five questions and everything demonstrated since, including this hour.

It is worth opening now and walking through once while the demonstrations are fresh.  The suggested use afterward: run down it once per result, before the result leaves the lab.  Most items cost one line of code or one honest sentence in a methods section.

<a id="homework"></a>

## 5. Homework: three diagnostics for your own data

Unlike the other sessions, this homework is deliberately **for after the workshop** — Three functions, defined in the code cell below that operationalize checklist items, meant to be pointed at your own datasets next week.  The check cells below verify them against this workshop's data.  Edit the function definitions in the code cell below: replace each `raise RuntimeError` line with your own code, then re-run that cell before running the checks.

### Your code goes here: `a14pm_hw2`

The functions below are the ones you need to write. **Edit them right here in this cell**, then re-run this cell (Shift+Enter) to update your definitions. Re-run the cells further down to test them.

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, KFold, cross_val_score


def grouped_split(X, y, groups, test_size=0.2):
    '''
    An honest train/test split: related samples never straddle the split.

    Use sklearn's GroupShuffleSplit with n_splits=1, the given test_size,
    and random_state=0 to get one (train_idx, test_idx) pair in which no
    group appears on both sides, then return the four arrays.

    @param:
    X (array, shape (n, d)), y (array, shape (n,))
    groups (array, shape (n,)): group ids (subject, family, batch, ...)
    test_size (float): fraction of samples in the test set

    @return:
    X_train, X_test, y_train, y_test

    Hint: next(GroupShuffleSplit(...).split(X, y, groups)) yields the two
    index arrays.
    '''
    raise RuntimeError("You need to write this part!")


def learning_curve(X, y, sizes):
    '''
    Is performance still climbing with more data, or has it plateaued?

    Shuffle the sample indices once with
    np.random.default_rng(0).permutation(len(y)).  For each n in `sizes`,
    take the first n shuffled samples and compute mean cross-validated
    accuracy with LogisticRegression(max_iter=2000) and
    KFold(5, shuffle=True, random_state=0).  Return the list of
    accuracies, in the order of `sizes`.

    Reading the result: still climbing at the largest n -> collecting more
    of the same data will help; flat -> it won't, and effort belongs
    elsewhere (better features, better labels, a different model).

    @param:
    X (array, shape (n, d)), y (array, shape (n,))
    sizes (list of int): sample sizes to evaluate

    @return:
    accs (list of float), same length as sizes
    '''
    raise RuntimeError("You need to write this part!")


def check_duplicates(X, threshold=0.998):
    '''
    Find near-duplicate pairs of samples in a representation -- the
    quiet destroyers of honest splits.

    Compute all pairwise cosine similarities (1 - cosine distance) and
    return the list of index pairs (i, j) with i < j whose similarity
    exceeds `threshold`.

    Run this on your own data before splitting -- and even when an
    official grouping variable exists, because groupings miss things
    (in this workshop's protein dataset, this function finds paralog
    pairs that sit in different UniRef50 clusters).

    @param:
    X (array, shape (n, d)): one representation per sample
    threshold (float): cosine-similarity cutoff for "near-duplicate"

    @return:
    pairs (list of (int, int) tuples): the offending index pairs

    Hint: squareform(pdist(X, 'cosine')) gives distances;
    np.triu_indices(n, 1) walks each pair once.
    '''
    raise RuntimeError("You need to write this part!")

In [ ]:
help(grouped_split)
help(learning_curve)
help(check_duplicates)

**Check `grouped_split`.**  Expected output:

```
train 354  test 96   groups overlap: False
```

In [ ]:
X_tr, X_te, y_tr, y_te = grouped_split(embeddings, y, families, test_size=0.2)

# same deterministic split, to check the group property:
from sklearn.model_selection import GroupShuffleSplit
tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(embeddings, y, families))
print('train', len(X_tr), ' test', len(X_te),
      '  groups overlap:', bool(set(families[tr_idx]) & set(families[te_idx])))

**Check `learning_curve`** on the protein probe.  Expected output:

```
[0.84, 0.87, 0.885, 0.898]
```

In [ ]:
print([float(round(a, 3)) for a in learning_curve(embeddings, y, [50, 100, 200, 400])])

**Check `check_duplicates`.**  On the protein embeddings at the default threshold it should find **13 pairs** — and, tellingly, two of them are paralog pairs sitting in *different* UniRef50 clusters (GSTA2/GSTA5 and SULT1A1/SULT1A4): near-duplicates the official grouping variable missed.  Expected output:

```
13 pairs found
cross-cluster pairs: [('GSTA2', 'GSTA5'), ('SULT1A1', 'SULT1A4')]
```

In [ ]:
pairs = check_duplicates(embeddings)
print(len(pairs), 'pairs found')
cross = [(genes[i], genes[j]) for i, j in pairs if families[i] != families[j]]
print('cross-cluster pairs:', cross)

<a id="next"></a>

## 6. Where to go next

The recipe this workshop leaves behind: **get the data into a matrix → borrow a representation from a foundation model → probe for the variable of interest → validate with honest splits and a simple baseline → run the checklist.**  Each step was practiced on real data this week, and each transfers to other data types by changing one loader and one tokenizer.

Going further:

* **Communities:** the model hubs' own forums, the bioinformatics/imaging communities for each modality, and — most valuable — the person down the hall who has already fought with your data type.
* **Reading model cards** is a skill; the good ones state training data, intended use, and known failure modes, which is most of what the checklist needs.
* **Asking for help:** bring the learning curve, the split description, and the baseline number.  Anyone who can help will ask for those three things first.

**References for this session** (all verified 2026-08-10):

* Zech JR, Badgeley MA, Liu M, Costa AB, Titano JJ, Oermann EK (2018). Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross-sectional study. *PLOS Medicine* 15(11): e1002683. [doi:10.1371/journal.pmed.1002683](https://doi.org/10.1371/journal.pmed.1002683)
* Winkler JK et al. (2019). Association between surgical skin markings in dermoscopic images and diagnostic performance of a deep learning convolutional neural network for melanoma recognition. *JAMA Dermatology* 155(10):1135–1141. [doi:10.1001/jamadermatol.2019.1735](https://doi.org/10.1001/jamadermatol.2019.1735)
* Geirhos R, Jacobsen J-H, Michaelis C, Zemel R, Brendel W, Bethge M, Wichmann FA (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence* 2:665–673. [doi:10.1038/s42256-020-00257-z](https://doi.org/10.1038/s42256-020-00257-z)
* Power A, Burda Y, Edwards H, Babuschkin I, Misra V (2022). Grokking: generalization beyond overfitting on small algorithmic datasets. [arXiv:2201.02177](https://arxiv.org/abs/2201.02177)
* Yarkoni T & Westfall J (2017). Choosing prediction over explanation in psychology: lessons from machine learning. *Perspectives on Psychological Science* 12(6):1100–1122. [doi:10.1177/1745691617693393](https://doi.org/10.1177/1745691617693393)

<a id="batch"></a>

## Appendix (optional): batch effects, live

The Zech mechanism — the model reads the *measurement*, not the biology — demonstrated on this afternoon's blood-cell embeddings.  The setup simulates a subtle assay: only 8 of the 384 embedding dimensions are kept (a deliberately weak measurement), samples are assigned to 4 processing "batches" that **correlate with cell type** (as real batches do, whenever samples are collected group by group), and each batch stamps its own offset onto the measurements:

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

img_emb = np.load('image_embeddings.npy')
labels_img = np.load('blood_cells.npz')['labels']

weak = img_emb[:, :8].copy()                    # a deliberately weak 8-number assay
rngb = np.random.default_rng(2)
batch = np.where(rngb.random(len(labels_img)) < 0.8, labels_img // 2,
                 rngb.integers(0, 4, len(labels_img)))   # batches confounded with class
offsets = rngb.normal(0, 1.0, (4, weak.shape[1])) * weak.std()
weak_batched = weak + offsets[batch]            # each batch stamps its signature

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
a = cross_val_score(pipe, weak, labels_img, cv=KFold(5, shuffle=True, random_state=0)).mean()
b = cross_val_score(pipe, weak_batched, labels_img, cv=KFold(5, shuffle=True, random_state=0)).mean()
c = cross_val_score(pipe, weak_batched, labels_img, cv=GroupKFold(4), groups=batch).mean()

print(f'clean weak assay, random split:      {a:.3f}')
print(f'batched assay, random split:         {b:.3f}   <- INFLATED: the model reads the batch')
print(f'batched assay, split by batch:       {c:.3f}   <- the honest, and devastating, number')

Three numbers, one story.  The batch stamps *raise* the random-split accuracy from 0.45 to 0.64 — the model happily uses the processing signature as if it were biology, and a random split can never catch it, because the same signatures appear on both sides.  Splitting by batch tells the truth twice over: the inflation vanishes, *and* the weak assay is exposed as nearly worthless once the batch is accounted for (0.03 — below chance, because the learned batch signatures actively mislead on unseen batches).

The defense is the same one-line habit as always — group the split by the thing that could leak — plus the experimental-design version: randomize samples across batches *at collection time*, so batch cannot correlate with the thing being predicted.  Statistics fixes what it can; design fixes it better.